In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [2]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

Latest run date: 2025-03-10 20:45:31.208657


#### Functions

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

#### Constants

In [4]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

# dict tiers
dict_tiers = {
    'A1': 0.0760,
    'A': 0.1320,
    'B': 0.2650,
    'C': 0.3220,
    'D': 0.3500, 
}

# factor 24 to 72
flt_factor_24_to_72 = 2.36

# current approval rate
flt_current_approval_rate = 0.20

Project: 20250307-funded-trends
Task: 05_get_gen12_predictions


#### Make output dir

In [5]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [6]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/04_join_targets/{str_filename}'
df = pd.read_parquet(str_uri)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltNetChgOff_2,fltNetChgOff_3,fltNetChgOff_6,fltNetChgOff_12,fltNetChgOff_24,co_at_60,co_at_90,co_at_180,co_at_360,co_at_720
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.0,0.0,0.0,0.0,7827.16,0.0,0.0,0.0,0.0,0.315976
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000


#### Gen 12 Predictions

In [7]:
# preprocess
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in list_str_filename:
    str_bucket_path = f'01_ad/02_model/noPTImodel10/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path,
        str_bucket_path=str_bucket_path,
        str_project='20231010-gen-xii',
    )
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

# rm pmt hx
list_cols = [col for col in df.columns if 'pmthx' in col] + ['list_institutions','applicationdayofweek__app']
for col in tqdm(list_cols):
    if col in list(df.columns):
        df.drop(col, axis=1, inplace=True)
    else:
        pass

# preprocess
df_tmp = cls_model_preprocessing.transform(df)
# rm
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    os.remove(str_filename)

100%|██████████| 42/42 [00:29<00:00,  1.40it/s]


NaN Replacer: 1.7399 sec.


100%|██████████| 3/3 [00:00<00:00, 46.70it/s]

Set strings: 0.065959 sec.


Boolean Replacer: 3.4071 sec.


...


100%|██████████| 2037/2037 [00:02<00:00, 928.48it/s] 


Replace inf and -inf with NaN: 2.6582 sec.
Imputer: 2.3667 sec.
Map term: 0.051603 sec.
Map PTI: 0.057195 sec.


100%|██████████| 9/9 [00:00<00:00, 1221.21it/s]


Round values: 0.011078 sec.
Preprocessing Model: 19.292 sec.


100%|██████████| 2/2 [00:00<00:00, 13662.23it/s]


In [8]:
# predict - PD
str_filename = 'final_model.pkl'
str_model = '02_pricing_pd'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['gen12_pd'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]

In [9]:
# predict - lgd
str_filename = 'final_model.pkl'
str_model = '03_pricing_lgd'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['gen12_lgd'] = cls_model_inference.predict(df[list_cols_model])

#### Save to s3

In [10]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)